# 49. MLOps, drift, reproducibility, and model artifacts

**Level:** Advanced

**Prerequisites:** 11-21 and 48: evaluation and serving.

[Course index](00_START_HERE.ipynb) | [Previous](48_Pydantic_FastAPI_and_Model_Serving.ipynb) | [Next](50_Fairness_Privacy_and_Responsible_AI.ipynb)

**Topics:** MLOps, Data drift, Concept drift, Model versioning, Model artifacts, Monitoring, Reproducibility.

**Matching script:** `examples\49_MLOps_Drift_and_Model_Artifacts.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

MLOps connects dataset versions, training code, preprocessing, evaluated artifacts, deployment,
monitoring, and rollback. Record feature schemas, units, seeds, package versions, training
provenance, and evaluation results. A model file alone is not enough to reproduce behavior.

Data drift changes the input distribution. Concept drift changes the relationship between
inputs and targets. A mean shift detector can flag one type of data change, but cannot prove
concept drift without appropriate outcome evidence. Not every input shift harms accuracy.

Monitor missingness, schema changes, latency, error rates, label quality, class balance, and
downstream costs. Define thresholds and escalation policies using validation/historical
evidence rather than arbitrary universal constants. Avoid automatic retraining loops that
ingest corrupted or adversarial data.

Artifact formats have security and compatibility implications. Never load untrusted pickle
or joblib files. JSON can safely represent simple numeric model parameters when schema and
preprocessing are controlled; complex frameworks need their documented safe loading practices.
Hashes detect a byte change only when compared against a trusted expected value.

## Mathematics or data contract

A simple standardized mean shift is abs(new_mean-reference_mean)/reference_std; it is not a universal drift test.

## Important limitation

The demonstration threshold is a teaching policy, not a statistically calibrated monitoring system.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Version a simple artifact and flag a shifted batch

**Purpose:** Keep provenance and monitoring separate from model inference.

### Read the code in this order
1. Serialize a small allowlisted model record.
2. Hash its canonical bytes.
3. Compare reference and new batch means using reference scale.
4. Assert that the synthetic large shift triggers an explicit flag.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
artifact SHA-256: ef15c4b7a67694ca6b260c00eed74fcbfdadc696ce783c023b15c1619adf29a8
synthetic drift flag: True shift: 4.079
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import hashlib
import json
import numpy as np

artifact = {"schema_version": 1, "model_version": "linear-v1",
            "feature_names": ["utilization"], "weights": [0.3], "bias": 5.0}
encoded = json.dumps(artifact, sort_keys=True, separators=(",", ":")).encode()
fingerprint = hashlib.sha256(encoded).hexdigest()
rng = np.random.default_rng(42)
reference = rng.normal(0, 1, 500)
new = rng.normal(4, 1, 200)
shift = abs(new.mean() - reference.mean()) / reference.std()
alert = {"feature": "utilization", "standardized_mean_shift": float(shift), "flagged": bool(shift > 2)}
assert len(fingerprint) == 64 and alert["flagged"]
assert json.loads(encoded)["feature_names"] == ["utilization"]
print("artifact SHA-256:", fingerprint)
print("synthetic drift flag:", alert["flagged"], "shift:", round(shift, 3))

artifact SHA-256: ef15c4b7a67694ca6b260c00eed74fcbfdadc696ce783c023b15c1619adf29a8
synthetic drift flag: True shift: 4.079


### Interpret example 1

The result should demonstrate this purpose: **Keep provenance and monitoring separate from model inference.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add feature missingness and prediction-rate monitoring, with an explicit alert record rather than silent retraining.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `MLOps` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A simple standardized mean shift is abs(new_mean-reference_mean)/reference_std; it is not a universal drift test.
- Name the boundary of the demonstration: The demonstration threshold is a teaching policy, not a statistically calibrated monitoring system.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.